# Home Credit Default Risk — Data Preprocessing

This notebook transforms the raw Home Credit application data into a
model-ready dataset for probability-of-default modeling.

The preprocessing strategy is informed by findings from the exploratory data
analysis and includes anomaly handling, missing-value treatment, feature
engineering, categorical encoding, and preparation of training and holdout
datasets.

The objective is to create a reproducible preprocessing workflow that can be
applied consistently to both training data and previously unseen applicants.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Project paths
PROJECT_PATH = "/content/drive/MyDrive/Home Credit Default Risk"
DATA_PATH = f"{PROJECT_PATH}/data"

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/application_train.csv")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

Rows: 307,511
Columns: 122


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [ ]:
df.shape

(307511, 122)

## 1. Modeling Setup

The dataset contains two columns with special roles:

- **TARGET:** Binary outcome indicating whether an applicant experienced
  repayment difficulty. This is the response variable for the probability
  of default model.
- **SK_ID_CURR:** Unique applicant identifier. This variable is retained for
  tracking applicants but excluded from model features because the arbitrary
  identifier should not contain meaningful predictive information.

The remaining variables will be evaluated and transformed into model features.

In [ ]:
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

X = df.drop(columns=[TARGET])
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

X shape: (307511, 121)
y shape: (307511,)

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Target distribution (%):
TARGET
0    91.93
1     8.07
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split temporary set equally into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (215257, 121)
Validation: (46127, 121)
Test: (46127, 121)


In [ ]:
split_summary = pd.DataFrame({
    "Observations": [
        len(y_train),
        len(y_val),
        len(y_test)
    ],
    "TARGET Rate": [
        y_train.mean(),
        y_val.mean(),
        y_test.mean()
    ]
}, index=["Train", "Validation", "Test"])

split_summary["TARGET Rate"] *= 100

split_summary

,Observations,TARGET Rate
Train,215257,8.072676
Validation,46127,8.073363
Test,46127,8.073363


## 2. Feature Engineering

Exploratory analysis identified several transformations that improve the
interpretability of the raw application variables and capture meaningful
financial relationships.

Feature engineering is implemented as a reusable function so that identical
transformations can be applied to the training, validation, test, and future
unseen datasets.

In [ ]:
def engineer_features(data):
    data = data.copy()

    # Applicant age
    data["AGE"] = -data["DAYS_BIRTH"] / 365.25

    # Employment anomaly identified during EDA
    data["EMPLOYMENT_ANOMALY"] = (
        data["DAYS_EMPLOYED"] == 365243
    ).astype(int)

    # Convert employment duration to years
    data["EMPLOYMENT_YEARS"] = np.where(
        data["DAYS_EMPLOYED"] == 365243,
        np.nan,
        -data["DAYS_EMPLOYED"] / 365.25
    )

    # Financial burden ratios
    data["CREDIT_INCOME_RATIO"] = (
        data["AMT_CREDIT"] / data["AMT_INCOME_TOTAL"]
    )

    data["ANNUITY_INCOME_RATIO"] = (
        data["AMT_ANNUITY"] / data["AMT_INCOME_TOTAL"]
    )

    data["CREDIT_ANNUITY_RATIO"] = (
        data["AMT_CREDIT"] / data["AMT_ANNUITY"]
    )

    return data

In [ ]:
X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (215257, 127)
Validation: (46127, 127)
Test: (46127, 127)


In [ ]:
X_train[
    [
        "AGE",
        "EMPLOYMENT_YEARS",
        "EMPLOYMENT_ANOMALY",
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "CREDIT_ANNUITY_RATIO"
    ]
].describe()

,AGE,EMPLOYMENT_YEARS,EMPLOYMENT_ANOMALY,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_ANNUITY_RATIO
count,215257.000000,176501.000000,215257.000000,215257.000000,215249.000000,215249.000000
mean,43.899766,6.533593,0.180045,3.957313,0.180830,21.624410
std,11.941273,6.407927,0.384226,2.687463,0.094677,7.816105
min,21.007529,0.000000,0.000000,0.004808,0.000224,8.036674
25%,33.973990,2.097194,0.000000,2.018667,0.114348,15.647569
50%,43.107461,4.514716,0.000000,3.265067,0.162629,20.000000
75%,53.889117,8.695414,0.000000,5.161600,0.228679,27.099985
max,68.993840,48.139630,1.000000,49.227200,1.451571,45.305079


In [ ]:
X_train["EMPLOYMENT_ANOMALY"].value_counts(normalize=True) * 100

,proportion
EMPLOYMENT_ANOMALY,
0,81.995475
1,18.004525


## 3. Feature Selection and Raw Variable Cleanup

Several raw variables are excluded before preprocessing because they have been
replaced with more interpretable engineered features or do not represent
meaningful applicant characteristics.

- `SK_ID_CURR` is a unique applicant identifier and is excluded from modeling.
- `DAYS_BIRTH` is replaced by applicant age in years (`AGE`).
- `DAYS_EMPLOYED` contains the anomalous value 365243 and is replaced by
  cleaned employment duration (`EMPLOYMENT_YEARS`) together with an
  `EMPLOYMENT_ANOMALY` indicator.

The remaining variables are retained initially. Further feature selection, if
necessary, will be performed using the training data rather than manually
removing variables based solely on exploratory observations.

In [ ]:
drop_cols = [
    "SK_ID_CURR",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED"
]

X_train = X_train.drop(columns=drop_cols)
X_val = X_val.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (215257, 124)
Validation: (46127, 124)
Test: (46127, 124)


In [ ]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))
print("Total:", len(numeric_cols) + len(categorical_cols))

Numeric features: 108
Categorical features: 16
Total: 124


In [ ]:
ratio_cols = [
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_ANNUITY_RATIO"
]

for col in ratio_cols:
    print(
        f"{col}: "
        f"Inf = {np.isinf(X_train[col]).sum()}, "
        f"NaN = {X_train[col].isna().sum()}"
    )

CREDIT_INCOME_RATIO: Inf = 0, NaN = 0
ANNUITY_INCOME_RATIO: Inf = 0, NaN = 8
CREDIT_ANNUITY_RATIO: Inf = 0, NaN = 8


In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": X_train.isnull().sum(),
    "Missing_Percent": X_train.isnull().mean() * 100
})

missing_summary = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Percent", ascending=False)
)

missing_summary.head(20)

,Missing_Count,Missing_Percent
COMMONAREA_MODE,150300,69.823513
COMMONAREA_AVG,150300,69.823513
COMMONAREA_MEDI,150300,69.823513
NONLIVINGAPARTMENTS_MODE,149354,69.384039
NONLIVINGAPARTMENTS_AVG,149354,69.384039
NONLIVINGAPARTMENTS_MEDI,149354,69.384039
FONDKAPREMONT_MODE,147099,68.336454
LIVINGAPARTMENTS_MODE,147049,68.313226
LIVINGAPARTMENTS_MEDI,147049,68.313226
LIVINGAPARTMENTS_AVG,147049,68.313226


## 4. Missing Value Treatment

Missing values are handled separately for numeric and categorical features.

For numeric variables, missing observations are replaced using the median
calculated from the training data. Median imputation is used because many
financial variables are skewed and contain extreme values.

Missing-value indicators are also created for numeric variables containing
missing observations. This allows the model to distinguish between an
originally observed value and one introduced through imputation.

For categorical variables, missing observations are assigned to a separate
"Missing" category.

All imputation parameters are learned exclusively from the training data and
then applied unchanged to validation and test data to prevent data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

## 5. Apply Preprocessing

The preprocessing pipeline is fitted exclusively on the training data.
This allows the imputation values, scaling parameters, missingness indicators,
and categorical levels to be learned without using information from the
validation or test sets.

The fitted preprocessing pipeline is then applied unchanged to the validation
and test datasets.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Training: (215257, 318)
Validation: (46127, 318)
Test: (46127, 318)


In [ ]:
print(
    "NaN in training:",
    np.isnan(X_train_processed.data).sum()
    if hasattr(X_train_processed, "data")
    else np.isnan(X_train_processed).sum()
)

print(
    "Inf in training:",
    np.isinf(X_train_processed.data).sum()
    if hasattr(X_train_processed, "data")
    else np.isinf(X_train_processed).sum()
)

NaN in training: 0
Inf in training: 0


## 6. Final Model Feature Space

After preprocessing, the original numeric and categorical variables are
converted into a fully numeric feature matrix.

The final feature space includes standardized numeric variables, missing-value
indicators, and one-hot encoded categorical variables. Feature names are
retained so that model coefficients and feature importance can later be
interpreted.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print("\nFirst 30 features:")

for feature in feature_names[:30]:
    print(feature)

Number of processed features: 318

First 30 features:
num__CNT_CHILDREN
num__AMT_INCOME_TOTAL
num__AMT_CREDIT
num__AMT_ANNUITY
num__AMT_GOODS_PRICE
num__REGION_POPULATION_RELATIVE
num__DAYS_REGISTRATION
num__DAYS_ID_PUBLISH
num__OWN_CAR_AGE
num__FLAG_MOBIL
num__FLAG_EMP_PHONE
num__FLAG_WORK_PHONE
num__FLAG_CONT_MOBILE
num__FLAG_PHONE
num__FLAG_EMAIL
num__CNT_FAM_MEMBERS
num__REGION_RATING_CLIENT
num__REGION_RATING_CLIENT_W_CITY
num__HOUR_APPR_PROCESS_START
num__REG_REGION_NOT_LIVE_REGION
num__REG_REGION_NOT_WORK_REGION
num__LIVE_REGION_NOT_WORK_REGION
num__REG_CITY_NOT_LIVE_CITY
num__REG_CITY_NOT_WORK_CITY
num__LIVE_CITY_NOT_WORK_CITY
num__EXT_SOURCE_1
num__EXT_SOURCE_2
num__EXT_SOURCE_3
num__APARTMENTS_AVG
num__BASEMENTAREA_AVG


In [ ]:
feature_type_summary = pd.Series({
    "Original numeric features": len(numeric_cols),
    "Original categorical features": len(categorical_cols),
    "Final processed features": len(feature_names)
})

feature_type_summary

,0
Original numeric features,108
Original categorical features,16
Final processed features,318


## 7. Save Preprocessing Outputs

The fitted preprocessing pipeline and transformed train, validation, and test
datasets are saved for use in subsequent modeling notebooks.

Saving the fitted preprocessor ensures that future observations can be
transformed using exactly the same imputation values, scaling parameters,
categorical mappings, and feature definitions learned from the training data.

In [ ]:
import joblib
from scipy import sparse

In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/Home Credit Default Risk")

(PROJECT_PATH / "models").mkdir(exist_ok=True)
(PROJECT_PATH / "data" / "processed").mkdir(parents=True, exist_ok=True)

In [ ]:
joblib.dump(
    preprocessor,
    PROJECT_PATH / "models" / "preprocessor.joblib"
)

['/content/drive/MyDrive/Home Credit Default Risk/models/preprocessor.joblib']

In [ ]:
sparse.save_npz(
    PROJECT_PATH / "data" / "processed" / "X_train.npz",
    sparse.csr_matrix(X_train_processed)
)

sparse.save_npz(
    PROJECT_PATH / "data" / "processed" / "X_val.npz",
    sparse.csr_matrix(X_val_processed)
)

sparse.save_npz(
    PROJECT_PATH / "data" / "processed" / "X_test.npz",
    sparse.csr_matrix(X_test_processed)
)

In [ ]:
y_train.to_csv(
    PROJECT_PATH / "data" / "processed" / "y_train.csv",
    index=False
)

y_val.to_csv(
    PROJECT_PATH / "data" / "processed" / "y_val.csv",
    index=False
)

y_test.to_csv(
    PROJECT_PATH / "data" / "processed" / "y_test.csv",
    index=False
)

In [ ]:
pd.Series(
    feature_names,
    name="feature"
).to_csv(
    PROJECT_PATH / "data" / "processed" / "feature_names.csv",
    index=False
)

In [ ]:
print("Processed data files:")
for file in (PROJECT_PATH / "data" / "processed").iterdir():
    print(" -", file.name)

print("\nModel files:")
for file in (PROJECT_PATH / "models").iterdir():
    print(" -", file.name)

Processed data files:
 - X_train.npz
 - X_val.npz
 - X_test.npz
 - y_train.csv
 - y_val.csv
 - y_test.csv
 - feature_names.csv

Model files:
 - preprocessor.joblib
